<a href="https://colab.research.google.com/github/aboettcher-sig/morocco-forests-sig/blob/main/scripts/utility/pull_planet_weekly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pull Planet weekly basemaps to Drive

Downloads Planet **weekly** basemap quads over an AOI into a Google Drive
folder, with metadata sidecars.

Flow (Planet Basemaps API): authenticate with the API key -> list **series**
and pick the weekly one -> list **mosaics** in that series within a date range
-> search **quads** intersecting the AOI bbox -> download each quad (Cloud
Optimized GeoTIFF) and write metadata.

Prerequisites in this Colab:
- A secret named `morroco_planet` holding the Planet API key
  (Colab left sidebar -> key icon -> add secret, notebook access on).
- The AOI as a `.gpkg` in Drive.
- A target Drive output folder.

Edit only the CONFIG cell.

In [4]:
!pip -q install geopandas pyogrio

In [5]:
import csv
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import requests
from shapely.geometry import box
from shapely.ops import unary_union

## CONFIG

The only cell to edit. Paths are inside `/content/drive/MyDrive/...`.
`SERIES_NAME_FILTER` is a case-insensitive substring used to find the weekly
series; `START`/`END` bound which weekly mosaics are pulled (by first-acquired
date, inclusive of START, exclusive of END).

In [6]:
SECRET_NAME = "morroco_planet"  # Colab secret holding the Planet API key (spelling matches the secret)

AOI_GPKG = "/content/drive/MyDrive/morocco_forests/test_region.gpkg"  # AOI polygon in Drive
AOI_LAYER = None                                       # gpkg layer name, or None for the first/only layer

OUT_DIR = "/content/drive/MyDrive/morocco_forests/planet_weekly"  # where quads + metadata are written

SERIES_NAME_FILTER = "weekly"   # case-insensitive substring to locate the weekly series
START = None            # inclusive, YYYY-MM-DD
END = None             # exclusive, YYYY-MM-DD

MIN_COVERAGE = 0.0    # keep quads with percent_covered >= this (0-100); 0 keeps all returned
EXACT_CLIP = True     # drop quads whose footprint misses the AOI polygon (no-op for a bbox AOI)
FETCH_PROVENANCE = True  # record contributing source scenes per quad (one extra API call each)
OVERWRITE = False     # re-download quads whose .tif already exists

## Authenticate

Planet uses HTTP basic auth with the API key as the username and an empty
password. All calls go through one session that carries the auth and a retry
wrapper for the quads rate limit (100 r/s).

In [7]:
from google.colab import userdata

try:
    API_KEY = userdata.get(SECRET_NAME)
except Exception as e:
    raise RuntimeError(
        f"Could not read Colab secret '{SECRET_NAME}'. Add it via the key icon "
        f"in the left sidebar and enable notebook access."
    ) from e

BASE = "https://api.planet.com/basemaps/v1"

session = requests.Session()
session.auth = (API_KEY, "")


def _get(url, params=None, stream=False):
    """GET with basic auth and backoff on 429 rate-limit responses."""
    for attempt in range(6):
        r = session.get(url, params=params, stream=stream, timeout=120)
        if r.status_code == 429:
            wait = float(r.headers.get("Retry-After", 2 ** attempt))
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    r.raise_for_status()
    return r


def paginate(url, item_key, params=None):
    """Yield items across pages, following _links._next."""
    while url:
        payload = _get(url, params=params).json()
        params = None  # _next already encodes the query
        for item in payload.get(item_key, []):
            yield item
        url = payload.get("_links", {}).get("_next")

## Mount Drive and load the AOI

The AOI is reprojected to EPSG:4326 (degrees) because the quad search takes a
lon/lat bounding box. `aoi_geom` is the unified footprint used for the optional
exact clip; `bbox_str` is the search rectangle.

In [8]:
from google.colab import drive

drive.mount("/content/drive",force_remount=True)

# Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

aoi = gpd.read_file(AOI_GPKG, layer=AOI_LAYER) if AOI_LAYER else gpd.read_file(AOI_GPKG)
aoi = aoi.to_crs(4326)
aoi_geom = unary_union(aoi.geometry.values)
minx, miny, maxx, maxy = aoi.total_bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"
print(f"AOI features: {len(aoi)}  bbox: {bbox_str}")

Mounted at /content/drive
AOI features: 1  bbox: -5.475904,35.399737,-5.465059,35.406735


## Find the weekly series

Lists all series visible to the key and filters by `SERIES_NAME_FILTER`. The
matches are printed so the choice is explicit; the first match is used. Set
`SERIES_ID` by hand below if more than one matches.

In [9]:
all_series = list(paginate(f"{BASE}/series/", "series"))
matches = [s for s in all_series if SERIES_NAME_FILTER.lower() in s.get("name", "").lower()]

print(f"{len(all_series)} series visible; {len(matches)} match '{SERIES_NAME_FILTER}':")
for s in matches:
    print(f"  {s['id']}  {s['name']}")

if not matches:
    raise RuntimeError(
        f"No series matched '{SERIES_NAME_FILTER}'. Available: "
        + ", ".join(sorted(s.get("name", "") for s in all_series))
    )

SERIES_ID = matches[0]["id"]
SERIES_NAME = matches[0]["name"]
print(f"\nUsing series: {SERIES_NAME} ({SERIES_ID})")

3 series visible; 1 match 'weekly':
  800f630f-9378-45dd-9028-dba68b70d841  PS sen2_normalized_analytic_8b_sr weekly subscription

Using series: PS sen2_normalized_analytic_8b_sr weekly subscription (800f630f-9378-45dd-9028-dba68b70d841)


## Select weekly mosaics in the date window

Pulls the mosaics belonging to the chosen series and keeps those whose
first-acquired date falls in [START, END). Each mosaic is one week.

In [11]:
def _parse(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))

start_dt = datetime.fromisoformat(START).replace(tzinfo=timezone.utc)
end_dt = datetime.fromisoformat(END).replace(tzinfo=timezone.utc)

# The documented /mosaics/{series_id}/mosaics/ path 404s. Use the mosaics link
# the series response carries, falling back to the /series/ path.
links = matches[0].get("_links", {})
mosaics_url = links.get("mosaics") or next(
    (v for k, v in links.items()
     if isinstance(v, str) and "/mosaics" in v and k != "_self"),
    f"{BASE}/series/{SERIES_ID}/mosaics/",
)

mosaics = list(paginate(mosaics_url, "mosaics"))
selected = [
    m for m in mosaics
    if start_dt <= _parse(m["first_acquired"]) < end_dt
]
selected.sort(key=lambda m: m["first_acquired"])

print(f"{len(mosaics)} mosaics in series; {len(selected)} within {START}..{END}:")
for m in selected:
    print(f"  {m['first_acquired'][:10]}  {m['name']}  ({m['id']})")

108 mosaics in series; 0 within 2024-01-01..2024-03-01:


## Download quads and write metadata

For each selected mosaic: search quads over the AOI bbox, optionally clip to
the AOI polygon and coverage threshold, download each quad GeoTIFF, and write
sidecars:

- `<mosaic_name>/_mosaic.json` - full mosaic metadata as returned by the API
- `<mosaic_name>/<quad_id>.tif` - the Cloud Optimized GeoTIFF
- `<mosaic_name>/<quad_id>.json` - quad bbox, percent_covered, download link,
  and contributing source scenes (provenance) when `FETCH_PROVENANCE`
- `manifest.csv` at the output root - one row per downloaded quad

In [13]:
selected = sorted(mosaics, key=lambda m: m["first_acquired"])

manifest_path = Path(OUT_DIR) / "manifest.csv"
rows, n_quads, n_fail = [], 0, 0


def quad_items(mosaic_id, quad_id):
    out = []
    for it in paginate(f"{BASE}/mosaics/{mosaic_id}/quads/{quad_id}/items", "items"):
        out.append(it.get("link", it.get("id", it)) if isinstance(it, dict) else it)
    return out


def download_quad(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with _get(url, stream=True) as r, open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            if chunk:
                f.write(chunk)
    tmp.replace(dest)


for m in selected:
    mosaic_id, mosaic_name = m["id"], m["name"]
    mdir = Path(OUT_DIR) / mosaic_name
    mdir.mkdir(parents=True, exist_ok=True)
    (mdir / "_mosaic.json").write_text(json.dumps(m, indent=2))

    quads = list(paginate(
        f"{BASE}/mosaics/{mosaic_id}/quads/", "items", params={"bbox": bbox_str}
    ))

    kept = 0
    for q in quads:
        cov = q.get("percent_covered", 100)
        if cov < MIN_COVERAGE:
            continue
        if EXACT_CLIP:
            x0, y0, x1, y1 = q["bbox"]
            if not box(x0, y0, x1, y1).intersects(aoi_geom):
                continue

        quad_id = q["id"]
        tif = mdir / f"{quad_id}.tif"
        try:
            if OVERWRITE or not tif.exists():
                download_quad(q["_links"]["download"], tif)
            scenes = quad_items(mosaic_id, quad_id) if FETCH_PROVENANCE else []
        except Exception as e:
            n_fail += 1
            print(f"  FAIL {mosaic_name}/{quad_id}: {type(e).__name__} {e}")
            continue

        meta = {
            "mosaic_id": mosaic_id, "mosaic_name": mosaic_name,
            "first_acquired": m["first_acquired"], "last_acquired": m["last_acquired"],
            "quad_id": quad_id, "bbox": q["bbox"], "percent_covered": cov,
            "download": q["_links"]["download"], "source_scenes": scenes,
        }
        (mdir / f"{quad_id}.json").write_text(json.dumps(meta, indent=2))
        rows.append({
            "mosaic_name": mosaic_name, "mosaic_id": mosaic_id,
            "first_acquired": m["first_acquired"], "last_acquired": m["last_acquired"],
            "quad_id": quad_id, "percent_covered": cov, "bbox": q["bbox"],
            "n_source_scenes": len(scenes), "tif_path": str(tif),
        })
        n_quads += 1
        kept += 1

    print(f"{mosaic_name}: {len(quads)} quads over bbox, {kept} kept")
    time.sleep(0.2)

if rows:
    with open(manifest_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

print(f"\nDone. {n_quads} quads, {n_fail} failed, across {len(selected)} mosaics -> {OUT_DIR}")
print(f"manifest: {manifest_path}")

ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-07-29_2024-08-05_mosaic: 1 quads over bbox, 1 kept
ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-05_2024-08-12_mosaic: 1 quads over bbox, 1 kept
ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-12_2024-08-19_mosaic: 1 quads over bbox, 1 kept
ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-19_2024-08-26_mosaic: 1 quads over bbox, 1 kept
ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-08-26_2024-09-02_mosaic: 1 quads over bbox, 1 kept
ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-09-02_2024-09-09_mosaic: 1 quads over bbox, 1 kept
ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-09-09_2024-09-16_mosaic: 1 quads over bbox, 1 kept
ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-09-16_2024-09-23_mosaic: 1 quads over bbox, 1 kept
ps_weekly_sen2_normalized_analytic_8b_sr_subscription_2024-09-23_2024-09-30_mosaic: 1 quads over bbox, 1 kept
ps_weekly_